# Assignment 2 - Preprocessing
**Name:** Manpreet Singh
**Subgroup:** 3c25
**Roll No:** 102303357



## Q1) Part I: Based on Feature Selection, Cleaning, and Preprocessing to Construct an Input from Data Source

(a) Examine the values of each attribute and Select a set of attributes only that would affect to predict
future bike buyers to create your input for data mining algorithms. Remove all the unnecessary
attributes. (Select features just by analysis).

We made `Age` attribute from `BirthDate`.  
The features selected are: `Age`, `Education`, `Occupation`, `Gender`, `MaritalStatus`, `HomeOwnerFlag`, `NumberCarsOwned`, `NumberChildrenAtHome`, `TotalChildren`, and `YearlyIncome`.  
All other attributes were removed.  


(b) Create a new Data Frame with the selected attributes only.

In [3]:
import pandas as pd

customers_df = pd.read_csv("https://raw.githubusercontent.com/DevManpreet5/MachineLearning/refs/heads/main/Assignment2/Dataset/AWCustomers.csv")
sales_df = pd.read_csv("https://raw.githubusercontent.com/DevManpreet5/MachineLearning/refs/heads/main/Assignment2/Dataset/AWSales.csv")

full_data = pd.merge(customers_df, sales_df, on="CustomerID")
full_data["BirthDate"] = pd.to_datetime(full_data["BirthDate"], errors="coerce")
full_data["Age"] = ((pd.Timestamp.today() - full_data["BirthDate"]).dt.days / 365).astype(int)

columns_to_keep = ["Age", "Education", "Occupation", "Gender", "MaritalStatus", "HomeOwnerFlag",
                   "NumberCarsOwned", "NumberChildrenAtHome", "TotalChildren", "YearlyIncome", "BikeBuyer"]
df = full_data[columns_to_keep]
df.head()


,Age,Education,Occupation,Gender,MaritalStatus,HomeOwnerFlag,NumberCarsOwned,NumberChildrenAtHome,TotalChildren,YearlyIncome,BikeBuyer
0,37,Bachelors,Clerical,M,M,1,3,0,1,81916,1
1,53,Partial College,Clerical,M,M,1,2,1,2,81076,1
2,39,Bachelors,Clerical,F,S,0,3,0,0,86387,1
3,47,Partial College,Skilled Manual,M,M,1,2,1,2,61481,1
4,50,Partial College,Skilled Manual,M,S,1,1,0,0,51804,1


c) Determine a Data value type (Discrete, or Continuous, then Nominal, Ordinal, Interval, Ratio) of
each attribute in your selection to identify preprocessing tasks to create input for your data mining. 

| Attribute             | Data Type   | Measurement Level |
|-----------------------|------------|-----------------|
| Age                   | Continuous | Ratio           |
| Education             | Discrete   | Ordinal         |
| Occupation            | Categorical | Nominal        |
| Gender                | Categorical | Nominal        |
| MaritalStatus         | Categorical | Nominal        |
| HomeOwnerFlag         | Binary     | Nominal         |
| NumberCarsOwned       | Discrete   | Ratio           |
| NumberChildrenAtHome  | Discrete   | Ratio           |
| TotalChildren         | Discrete   | Ratio           |
| YearlyIncome          | Continuous | Ratio           |
| BikeBuyer             | Binary     | Nominal         |


## Q2) Data Preprocessing and Transformation
Depending on the data type of each attribute, transform each object from your preprocessed data. 
Use all the data rows (~= 18000 rows) with the selected features as input to apply all the tasks below, do
not perform each task on the smaller data set that you got from your random sampling result.

(a) Handling Null values

In [ ]:
numeric_cols = ["Age","NumberCarsOwned","NumberChildrenAtHome","TotalChildren","YearlyIncome"]
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

categorical_cols = ["Education","Occupation","Gender","MaritalStatus","HomeOwnerFlag"]
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

(b) Normalization

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
scaler = MinMaxScaler()
df[["Age","YearlyIncome"]] = scaler.fit_transform(df[["Age","YearlyIncome"]])

(c) Discretization (Binning) on Continuous attributes or Categorical Attributes with too many different
values

In [ ]:
df["AgeGroup"] = pd.cut(df["Age"], bins=[0,0.25,0.5,0.75,1.0], labels=["Young","Adult","Middle-Age","Senior"])
df["IncomeGroup"] = pd.qcut(df["YearlyIncome"], q=4, labels=["Low","Medium","High","Very High"])

(d) Standardization/Normalization

In [ ]:
scaler_std = StandardScaler()
df[["NumberCarsOwned","NumberChildrenAtHome","TotalChildren"]] = scaler_std.fit_transform(
    df[["NumberCarsOwned","NumberChildrenAtHome","TotalChildren"]]
)

(e) Binarization (One Hot Encoding) 

In [9]:
df = pd.get_dummies(df, columns=["Education","Occupation","Gender","MaritalStatus","HomeOwnerFlag","AgeGroup","IncomeGroup"], drop_first=True)

In [10]:
print(df.head())

        Age  NumberCarsOwned  NumberChildrenAtHome  TotalChildren  \
0  0.183099         1.892524             -0.594371       0.161342   
1  0.408451         0.798389              1.163279       1.239753   
2  0.211268         1.892524             -0.594371      -0.917069   
3  0.323944         0.798389              1.163279       1.239753   
4  0.366197        -0.295746             -0.594371      -0.917069   

   YearlyIncome  BikeBuyer  Education_Graduate Degree  Education_High School  \
0      0.496842          1                      False                  False   
1      0.489453          1                      False                  False   
2      0.536172          1                      False                  False   
3      0.317083          1                      False                  False   
4      0.231958          1                      False                  False   

   Education_Partial College  Education_Partial High School  ...  \
0                      False        

## Q3) Calculating Proximity /Correlation Analysis of two features 

(a) Calculate Similarity in Simple Matching, Jaccard Similarity, and Cosine Similarity between two
following objects of your transformed input data. 

In [14]:
import numpy as np
from sklearn.metrics import jaccard_score
from sklearn.metrics.pairwise import cosine_similarity

obj1 = df.iloc[0]
obj2 = df.iloc[1]

numeric_cols = ["Age", "NumberCarsOwned", "NumberChildrenAtHome", "TotalChildren", "YearlyIncome"]
binary_cols = [col for col in df.columns if col not in numeric_cols + ["BikeBuyer"]]

obj1_num = obj1[numeric_cols].values.reshape(1, -1)
obj2_num = obj2[numeric_cols].values.reshape(1, -1)
cos_sim_numeric = cosine_similarity(obj1_num, obj2_num)[0][0]

obj1_bin = obj1[binary_cols].astype(int).values
obj2_bin = obj2[binary_cols].astype(int).values
jaccard_sim = jaccard_score(obj1_bin, obj2_bin)
simple_match = np.mean(obj1_bin == obj2_bin)

print("Similarity between Object 1 and Object 2:")
print(f"Cosine similarity (numeric features): {cos_sim_numeric:.3f}")
print(f"Simple Matching similarity (binary features): {simple_match:.3f}")
print(f"Jaccard similarity (binary features): {jaccard_sim:.3f}")


Similarity between Object 1 and Object 2:
Cosine similarity (numeric features): 0.327
Simple Matching similarity (binary features): 0.882
Jaccard similarity (binary features): 0.600


(b) Calculate Correlation between two features Commute Distance and Yearly Income 

In [15]:
if "CommuteDistance" in df.columns:
    corr = df["CommuteDistance"].corr(df["YearlyIncome"])
    print(f"\nCorrelation between CommuteDistance and YearlyIncome: {corr:.3f}")
else:
    print("\nColumn 'CommuteDistance' not found in dataset.")



Column 'CommuteDistance' not found in dataset.
